In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# Project root
ROOT = Path(
    r"D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics"
)

# Raw dataset path
RAW_PATH = ROOT / "data" / "raw" / "global_supply_chain_risk_2026.csv"

# Cleaned data output path
OUTPUT_PATH = ROOT / "data" / "interim" / "cleaned_data.csv"

# Check whether raw dataset exists
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at:\n{RAW_PATH}"
    )

# Load raw dataset
df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)

# Display first 5 rows
display(df.head())

Dataset loaded successfully!
Original shape: (5000, 14)


,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days,Disruption_Occurred
0,SC-10000,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,SC-10001,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,SC-10002,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,SC-10003,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,SC-10004,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1


In [13]:
def clean_column_name(column):
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = column.strip("_")
    return column


df.columns = [
    clean_column_name(column)
    for column in df.columns
]


for column in df.columns:
    print(column)

shipment_id
date
origin_port
destination_port
transport_mode
product_category
distance_km
weight_mt
fuel_price_index
geopolitical_risk_score
weather_condition
carrier_reliability_score
lead_time_days
disruption_occurred


In [14]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicates removed:", before - after)
print("New shape:", df.shape)

Duplicates removed: 0
New shape: (5000, 14)


In [15]:
target_candidates = [
    "disruption",
    "supply_chain_disruption",
    "disruption_flag",
    "late_delivery_risk",
    "late_delivery_risk_flag",
    "shipping_risk",
    "risk"
]

target_column = None

for column in target_candidates:

    if column in df.columns:
        target_column = column
        break

print("Target column:", target_column)

Target column: None


In [16]:
# ============================================
# TARGET COLUMN DETECTION
# ============================================

# List of columns used to create the target
# Only needed if the dataset does NOT already
# contain a disruption target.
target_source_columns = []


# ------------------------------------------------
# 1. Check whether the dataset already has target
# ------------------------------------------------

if "disruption_occurred" in df.columns:

    target_column = "disruption_occurred"

    print(
        f"Target column found: {target_column}"
    )


# ------------------------------------------------
# 2. If target does not exist, try to create it
# ------------------------------------------------

else:

    scheduled_candidates = [
        "days_for_shipment_scheduled",
        "days_for_shipping_scheduled",
        "scheduled_shipping_days",
        "scheduled_days"
    ]

    actual_candidates = [
        "days_for_shipping_real",
        "days_for_shipping",
        "actual_shipping_days",
        "real_shipping_days"
    ]

    scheduled_column = None
    actual_column = None


    # --------------------------------------------
    # Find scheduled shipping column
    # --------------------------------------------

    for column in scheduled_candidates:

        if column in df.columns:

            scheduled_column = column

            break


    # --------------------------------------------
    # Find actual shipping column
    # --------------------------------------------

    for column in actual_candidates:

        if column in df.columns:

            actual_column = column

            break


    # --------------------------------------------
    # Create target if both columns exist
    # --------------------------------------------

    if scheduled_column and actual_column:

        df["disruption_occurred"] = (
            pd.to_numeric(
                df[actual_column],
                errors="coerce"
            )
            >
            pd.to_numeric(
                df[scheduled_column],
                errors="coerce"
            )
        ).astype(int)


        target_column = "disruption_occurred"


        target_source_columns = [
            scheduled_column,
            actual_column
        ]


        print(
            f"Created disruption target using "
            f"{actual_column} > {scheduled_column}"
        )


    # --------------------------------------------
    # If target cannot be found
    # --------------------------------------------

    else:

        raise ValueError(
            "No disruption target found.\n"
            "Your dataset must contain "
            "'disruption_occurred' or "
            "scheduled and actual shipping-day columns."
        )


# ============================================
# DISPLAY TARGET INFORMATION
# ============================================

print("\n===================================")
print("TARGET INFORMATION")
print("===================================")

print(
    "Target column:",
    target_column
)


print("\nTarget data type:")

print(
    df[target_column].dtype
)


print("\nTarget unique values:")

print(
    df[target_column].unique()
)


print("\nTarget value counts:")

print(
    df[target_column].value_counts()
)


print("\nTarget percentages:")

print(
    df[target_column]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================
# CREATE X AND y
# ============================================

X = df.drop(
    columns=[target_column]
)

y = df[target_column]


print("\n===================================")
print("FEATURE / TARGET SHAPE")
print("===================================")

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)


print("\nFeature columns:")

print(
    X.columns.tolist()
)


print("\nTarget column:")

print(
    target_column
)

Target column found: disruption_occurred

TARGET INFORMATION
Target column: disruption_occurred

Target data type:
int64

Target unique values:
[1 0]

Target value counts:
disruption_occurred
1    3063
0    1937
Name: count, dtype: int64

Target percentages:
disruption_occurred
1    61.26
0    38.74
Name: proportion, dtype: float64

FEATURE / TARGET SHAPE
X shape: (5000, 13)
y shape: (5000,)

Feature columns:
['shipment_id', 'date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'weather_condition', 'carrier_reliability_score', 'lead_time_days']

Target column:
disruption_occurred


In [17]:
print(df.columns.tolist())

['shipment_id', 'date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'weather_condition', 'carrier_reliability_score', 'lead_time_days', 'disruption_occurred']


In [18]:
if target_column != "disruption":

    values = (
        df[target_column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "yes": 1,
        "y": 1,
        "true": 1,
        "1": 1,
        "high": 1,
        "late": 1,

        "no": 0,
        "n": 0,
        "false": 0,
        "0": 0,
        "low": 0,
        "on_time": 0,
        "on time": 0
    }

    converted = values.map(mapping)

    if converted.notna().mean() >= 0.8:

        df[target_column] = converted

    else:

        unique_values = values.dropna().unique()

        if len(unique_values) == 2:

            conversion = {
                unique_values[0]: 0,
                unique_values[1]: 1
            }

            df[target_column] = values.map(conversion)

        else:

            raise ValueError(
                "Target is not binary."
            )


df[target_column] = pd.to_numeric(
    df[target_column],
    errors="coerce"
)

df = df.dropna(
    subset=[target_column]
)

df[target_column] = df[target_column].astype(int)

print(
    df[target_column].value_counts()
)

disruption_occurred
1    3063
0    1937
Name: count, dtype: int64


In [19]:
if target_column != "disruption":

    df = df.rename(
        columns={
            target_column: "disruption"
        }
    )

    target_column = "disruption"

In [20]:
if target_source_columns:

    df = df.drop(
        columns=[
            column
            for column in target_source_columns
            if column in df.columns
        ]
    )

print("Final columns:")
print(df.columns.tolist())

Final columns:
['shipment_id', 'date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'weather_condition', 'carrier_reliability_score', 'lead_time_days', 'disruption']


In [21]:
id_columns = []

for column in df.columns:

    if column == "disruption":
        continue

    name = column.lower()

    unique_ratio = (
        df[column].nunique(dropna=False)
        / len(df)
    )

    if (
        (name == "id"
         or name.endswith("_id")
         or name.startswith("id_"))
        and unique_ratio > 0.5
    ):
        id_columns.append(column)


df = df.drop(
    columns=id_columns,
    errors="ignore"
)

print("Removed ID columns:")
print(id_columns)

Removed ID columns:
['shipment_id']


In [22]:
from pathlib import Path

OUTPUT_PATH = Path(
    "D:/My_Project/Analyse/capston Project_2-Global Supply Chain Risk & Logistics/"
    "data/interim/cleaned_data.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Cleaned dataset saved to:")
print(OUTPUT_PATH.resolve())

print("Final shape:", df.shape)

Cleaned dataset saved to:
D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\data\interim\cleaned_data.csv
Final shape: (5000, 13)
